# Ground-pass DLOI-Pose demonstration
Exploratory heuristic, not calibrated probabilities or empirical validation. Pose z is centroid-relative: aerial scoring is unsupported. Neck-to-nose direction is not eye gaze.
Install with `python -m pip install -e '.[notebook]'` from the repository first. This notebook uses only the packaged sample and never downloads a match.
The roster and movement filter below are specific to this short match-1925299 sample, not full-match lineup inference. Centered velocities are retrospective.

In [ ]:
from pathlib import Path
import sys
root = Path.cwd().resolve()
if root.name == 'notebooks':
    root = root.parent
if (root / 'src').is_dir() and str(root) not in sys.path:
    sys.path.insert(0, str(root))
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from src.data.pose_loading import iter_sample
from src.data.data_loader import MATCH_1925299_ROSTER, identify_active_players, compute_frame_velocities
from src.features import DefensiveLaneOcclusion, shoulder_table, gating_report
from src.visualization.pitch_viz import plot_defensive_lanes
frames = list(iter_sample())
active = identify_active_players(frames)
model = DefensiveLaneOcclusion()
display(gating_report(iter(frames)))

In [ ]:
def defense(frame, index):
    players = [p for p in frame['player_data'] if p['player_id'] in active
               and MATCH_1925299_ROSTER[p['player_id']]['team_name'].startswith('Perth')
               and MATCH_1925299_ROSTER[p['player_id']]['position_group'] != 'Goalkeeper']
    velocities = compute_frame_velocities(frames, index)
    xy = np.array([[p['x'], p['y']] for p in players], dtype=float).reshape(-1, 2)
    vel = np.array([velocities.get(p['player_id'], np.zeros(2)) for p in players]).reshape(-1, 2)
    return players, xy, vel
targets = {'Central space': (np.array([25., 0.]), False),
           'Wide space': (np.array([22., 24.5]), False),
           'Backpass space': (np.array([-5.83, -2.71]), False)}
frame = frames[0]
ball = np.array([frame['ball_data']['x'], frame['ball_data']['y']])
players, xy, vel = defense(frame, 0)
rows = []
plot_targets = {}
for name, (target, _) in targets.items():
    baseline = model.compute_frame_occlusion(ball, target, xy, vel, players, use_pose=False)
    score = model.compute_frame_occlusion(ball, target, xy, vel, players)
    rows.append({'lane': name, 'isotropic': baseline, 'pose_proxy': score})
    plot_targets[name] = (target, score)
display(pd.DataFrame(rows))
display(pd.DataFrame(model.compute_counterfactual_attribution(ball, targets, xy, vel, players)))

In [ ]:
temporal = []
for index in range(0, min(100, len(frames)), 2):
    current = frames[index]
    current_ball = np.array([current['ball_data']['x'], current['ball_data']['y']])
    current_players, current_xy, current_vel = defense(current, index)
    row = {'t_sec': (current['frame'] - frame['frame']) / 25.}
    for name, (target, _) in targets.items():
        row[name] = model.compute_frame_occlusion(current_ball, target, current_xy, current_vel, current_players)
    temporal.append(row)
carrier_id = (frame.get('possession') or {}).get('player_id')
carrier = next((p for p in frame['player_data'] if p['player_id'] == carrier_id), None)
carrier_xy = np.array([carrier['x'], carrier['y']]) if carrier else ball
fig = plot_defensive_lanes(carrier_xy, ball, xy, [model.extract_pose_heading(p) for p in players],
    plot_targets, defenders_names=[MATCH_1925299_ROSTER[p['player_id']]['short_name'] for p in players],
    defenders_detected=[p.get('is_detected', False) for p in players],
    temporal_df=pd.DataFrame(temporal), title_suffix=f" | Frame {frame['frame']}")
plt.show()

Scores use fixed illustrative targets, not inferred intended passes. Leave-one-out values are nonadditive and can be zero for redundant defenders. No season rank, causal scanning relationship, or aerial advantage is inferred. See the project README for the exact algorithm and remaining limitations.